<a href="https://colab.research.google.com/github/Chathuranga630/ML-Workshop_SLT/blob/main/UAV_Fleet_Coo_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#!/usr/bin/env python3
"""
uav_fleet_sim.py - basic UAV fleet coordination simulation (stdlib only)

Mission: N UAVs must complete M waypoint tasks (fly there, hover for a service
time) before per-task deadlines, using as little energy as possible.

Coordinator
  * greedy energy-aware assignment: every idle UAV / pending task pair is scored
    by estimated energy (+ an urgency term), infeasible pairs (deadline or
    battery) are dropped, cheapest pairs are matched first.
  * optional defense (trust-based): detects unreliable UAVs, reassigns their
    tasks, blacklists them.

Attacks (a few UAVs are malicious)
  false_battery : reports a (nearly frozen) high battery level while really
                  being low -> wins tasks it cannot finish, dies mid-flight
  spoof_msg     : flies toward the task but sends a false "task complete"
                  message without doing the work, then asks for more
  non_coop      : accepts tasks but stalls (never flies / never finishes)

Modelling assumptions (all easy to change below)
  * The coordinator can physically observe UAV position / activity (radar or
    ground tracking), but NOT the battery level - that is self-reported.
  * Idle UAVs are landed (0 W). Single message link, no packet loss.
  * Audits (verifying a completion claim) are free but sampled with p_audit.

Usage
  python uav_fleet_sim.py                              # comparison table
  python uav_fleet_sim.py --seeds 100 --uavs 10 --tasks 20
  python uav_fleet_sim.py --scenario spoof_msg --defense --seed 3   # event log
"""
import argparse
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

# ----------------------------------------------------------------- constants
SPEED = 10.0            # m/s
P_CRUISE = 120.0        # W while flying
P_SERVICE = 90.0        # W while hovering to do the task
P_IDLE = 0.0            # W while idle (landed)
BATT_CAP = 20000.0      # J (full battery)
BASE = (250.0, 250.0)
AREA = 500.0            # m (square side)
DT = 1.0                # s

BATT_RESERVE = 0.05     # fraction of capacity kept as safety margin
BATT_TOL = 0.06         # reported-vs-expected battery gap that raises a flag
TRUST_MIN = 0.35        # blacklist below this


def dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


@dataclass
class Config:
    n_uavs: int = 8
    n_tasks: int = 16
    deadline: float = 400.0
    service_time: float = 10.0
    attack: str = "none"        # none | false_battery | spoof_msg | non_coop | mixed
    n_malicious: int = 2
    defense: bool = True
    p_audit: float = 0.5        # chance a completion claim is verified
    noncoop_prob: float = 0.8   # chance a non_coop UAV stalls on an assignment
    seed: int = 0
    verbose: bool = False


@dataclass
class Task:
    id: int
    pos: Tuple[float, float]
    deadline: float
    service_time: float
    status: str = "pending"     # pending | assigned | claimed | verified (coordinator's view)
    assigned_to: Optional[int] = None
    claimed_by: Optional[int] = None
    true_done: bool = False     # ground truth
    true_done_at: Optional[float] = None


# --------------------------------------------------------------------- UAV
class UAV:
    def __init__(self, uid: int, behavior: str, cfg: Config):
        self.id = uid
        self.behavior = behavior
        self.cfg = cfg
        self.rng = random.Random(cfg.seed * 1000 + uid)
        self.pos = BASE
        if behavior == "false_battery":
            self.batt = self.rng.uniform(0.15, 0.30) * BATT_CAP     # really low...
            self.fake_batt = self.rng.uniform(0.85, 1.0) * BATT_CAP  # ...claims high
        else:
            self.batt = self.rng.uniform(0.7, 1.0) * BATT_CAP
            self.fake_batt = 0.0
        self.state = "idle"     # idle | moving | servicing | stalled | dead
        self.task: Optional[Task] = None
        self.service_left = 0.0
        self.claim_time = 0.0   # spoofer only
        self.alive = True
        self.energy_used = 0.0
        self.power = 0.0        # power drawn in the last step (observable activity)

    def report_battery(self) -> float:
        base = self.fake_batt if self.behavior == "false_battery" else self.batt
        return max(0.0, base + self.rng.gauss(0, 0.005 * BATT_CAP))

    def accept(self, task: Task, now: float):
        self.task = task
        travel = dist(self.pos, task.pos) / SPEED
        if self.behavior == "non_coop" and self.rng.random() < self.cfg.noncoop_prob:
            self.state = "stalled"
        else:
            self.state = "moving"
        if self.behavior == "spoof_msg":
            # claims completion at 30%..110% of the honest completion time
            self.claim_time = now + self.rng.uniform(0.3, 1.1) * (travel + task.service_time)

    def cancel(self):
        if self.alive:
            self.task = None
            self.state = "idle"

    def step(self, now: float) -> List[tuple]:
        msgs: List[tuple] = []
        self.power = 0.0
        if not self.alive:
            return msgs

        if self.behavior == "spoof_msg" and self.task is not None and now >= self.claim_time:
            msgs.append(("claim", self.id, self.task.id))   # false: no work done
            self.task = None
            self.state = "idle"

        if self.state == "moving":
            tgt = self.task.pos
            d = dist(self.pos, tgt)
            step = SPEED * DT
            if d <= step:
                self.pos = tgt
                self.state = "servicing"
                self.service_left = self.task.service_time
            else:
                self.pos = (self.pos[0] + (tgt[0] - self.pos[0]) * step / d,
                            self.pos[1] + (tgt[1] - self.pos[1]) * step / d)
            self.power = P_CRUISE
        elif self.state == "servicing":
            self.power = P_SERVICE
            if self.behavior != "spoof_msg":                # spoofer just hovers
                self.service_left -= DT
                if self.service_left <= 0:
                    t = self.task
                    if not t.true_done:
                        t.true_done, t.true_done_at = True, now
                    msgs.append(("claim", self.id, t.id))
                    self.task = None
                    self.state = "idle"
        else:                                               # idle / stalled
            self.power = P_IDLE

        e = self.power * DT
        self.energy_used += e
        self.batt -= e
        if self.behavior == "false_battery":
            self.fake_batt -= 0.2 * e                       # lies about the drain
        if self.batt <= 0:
            self.alive, self.state, self.task = False, "dead", None
        return msgs


# ------------------------------------------------------------- coordinator
class Coordinator:
    def __init__(self, uavs: List[UAV], tasks: List[Task], cfg: Config):
        self.uavs: Dict[int, UAV] = {u.id: u for u in uavs}
        self.tasks = tasks
        self.cfg = cfg
        self.rng = random.Random(cfg.seed + 777)
        self.trust = {i: 1.0 for i in self.uavs}
        self.blacklist: Dict[int, float] = {}               # uid -> time banned
        self.info: Dict[int, dict] = {}                     # uid -> current assignment info
        self.model = {i: u.report_battery() for i, u in self.uavs.items()}
        self.last_flag = {i: -1e9 for i in self.uavs}
        self.last_seen = {i: 0.0 for i in self.uavs}
        self.done = False
        self.false_claims_accepted = 0

    # ---- logging
    def log(self, now, msg):
        if self.cfg.verbose:
            print(f"[t={now:5.0f}s] {msg}")

    # ---- main step
    def step(self, now: float, msgs: List[tuple]):
        for uid, u in self.uavs.items():
            self.model[uid] -= u.power * DT                 # expected battery from observed activity
            if u.alive:
                self.last_seen[uid] = now
        for _, uid, tid in msgs:
            self.on_claim(uid, tid, now)
        if self.cfg.defense:
            self.monitor(now)
        self.assign(now)
        self.check_done(now)

    # ---- assignment: greedy, energy + time aware
    def assign(self, now: float):
        idle = [u for u in self.uavs.values()
                if u.id not in self.info and u.id not in self.blacklist
                and not (self.cfg.defense and now - self.last_seen[u.id] > 5)]
        pending = [t for t in self.tasks if t.status == "pending"]
        if not idle or not pending:
            return
        cands = []
        for u in idle:
            batt = u.report_battery()                        # self-reported!
            for t in pending:
                travel = dist(u.pos, t.pos) / SPEED
                finish = now + travel + t.service_time
                if finish > t.deadline:
                    continue
                energy = travel * P_CRUISE + t.service_time * P_SERVICE
                ret = dist(t.pos, BASE) / SPEED * P_CRUISE
                if batt - energy - ret < BATT_RESERVE * BATT_CAP:
                    continue
                slack = t.deadline - finish
                score = energy + 20.0 * max(0.0, 100.0 - slack)   # energy + urgency
                if self.cfg.defense:
                    score /= max(self.trust[u.id], 0.1)            # prefer trusted UAVs
                cands.append((score, u, t))
        cands.sort(key=lambda c: c[0])
        used_u, used_t = set(), set()
        for score, u, t in cands:
            if u.id in used_u or t.id in used_t:
                continue
            used_u.add(u.id)
            used_t.add(t.id)
            t.status, t.assigned_to = "assigned", u.id
            self.info[u.id] = dict(task=t.id, t0=now, p0=u.pos,
                                   d0=dist(u.pos, t.pos), checked=False)
            u.accept(t, now)
            self.log(now, f"assign task {t.id} -> UAV {u.id} (score {score:.0f})")

    # ---- completion claims
    def on_claim(self, uid: int, tid: int, now: float):
        t = self.tasks[tid]
        inf = self.info.get(uid)
        if t.status in ("claimed", "verified"):
            if inf and inf["task"] == tid:
                self.info.pop(uid)
            return
        verified = False
        if self.cfg.defense:
            if inf is None or inf["task"] != tid:
                self.penalize(uid, 0.5, now, "unsolicited completion claim")
                return
            min_time = inf["d0"] / SPEED + 0.9 * t.service_time
            if now - inf["t0"] < min_time:
                self.penalize(uid, 0.5, now,
                              f"implausible completion time on task {tid}", release=True)
                return
            if self.rng.random() < self.cfg.p_audit:
                if not t.true_done:
                    self.penalize(uid, 0.5, now, f"audit failed on task {tid}", release=True)
                    return
                verified = True
                self.trust[uid] = min(1.0, self.trust[uid] + 0.05)
        if not t.true_done:
            self.false_claims_accepted += 1
        self.info.pop(uid, None)
        t.status = "verified" if verified else "claimed"
        t.claimed_by, t.assigned_to = uid, None
        self.log(now, f"UAV {uid} completed task {tid} ({'audited' if verified else 'accepted'})")

    # ---- monitoring (defense)
    def monitor(self, now: float):
        for uid, u in self.uavs.items():
            if uid in self.blacklist:
                continue
            if now - self.last_seen[uid] > 5:
                self.penalize(uid, 1.0, now, "telemetry lost (UAV down)")
                continue
            gap = u.report_battery() - self.model[uid]
            if gap > BATT_TOL * BATT_CAP and now - self.last_flag[uid] >= 10:
                self.last_flag[uid] = now
                self.penalize(uid, 0.3, now,
                              f"battery report inconsistent (+{gap:.0f} J vs model)")
                if uid in self.blacklist:
                    continue
            inf = self.info.get(uid)
            if inf:
                t = self.tasks[inf["task"]]
                el = now - inf["t0"]
                d = dist(u.pos, t.pos)
                if not inf["checked"] and el >= 15:
                    inf["checked"] = True
                    if d > 1.0 and d > inf["d0"] - 0.5 * SPEED * el:
                        self.penalize(uid, 0.5, now, f"no progress toward task {t.id}", release=True)
                        continue
                if el > 1.5 * (inf["d0"] / SPEED + t.service_time) + 15:
                    self.penalize(uid, 0.5, now, f"timeout on task {t.id}", release=True)

    # ---- trust / reassignment helpers
    def release(self, uid: int):
        inf = self.info.pop(uid, None)
        if inf:
            t = self.tasks[inf["task"]]
            if t.status == "assigned":
                t.status, t.assigned_to = "pending", None
        self.uavs[uid].cancel()

    def penalize(self, uid: int, amount: float, now: float, why: str, release=False):
        self.trust[uid] = max(0.0, self.trust[uid] - amount)
        self.log(now, f"UAV {uid} penalized ({why}); trust={self.trust[uid]:.2f}")
        if release:
            self.release(uid)
        if self.trust[uid] < TRUST_MIN and uid not in self.blacklist:
            self.ban(uid, now)

    def ban(self, uid: int, now: float):
        self.blacklist[uid] = now
        if uid in self.info:
            self.release(uid)
        reverted = 0
        for t in self.tasks:                                 # retroactive audit
            if t.claimed_by == uid and t.status == "claimed" and not t.true_done:
                t.status, t.claimed_by = "pending", None
                reverted += 1
        self.log(now, f"UAV {uid} BLACKLISTED; reassigning its work "
                      f"({reverted} unverified completed task(s) reverted)")

    def check_done(self, now: float):
        if not all(t.status in ("claimed", "verified") for t in self.tasks):
            return
        if self.cfg.defense:                                 # final audit before declaring success
            reverted = False
            for t in self.tasks:
                if t.status == "claimed":
                    if t.true_done:
                        t.status = "verified"
                    else:
                        cb = t.claimed_by
                        t.status, t.claimed_by = "pending", None
                        reverted = True
                        self.penalize(cb, 0.5, now, f"final audit failed on task {t.id}")
            if reverted:
                return
        self.done = True


# ------------------------------------------------------------------- driver
ATTACKS = {
    "none": [],
    "false_battery": ["false_battery"],
    "spoof_msg": ["spoof_msg"],
    "non_coop": ["non_coop"],
    "mixed": ["false_battery", "spoof_msg", "non_coop"],
}


def run_sim(cfg: Config) -> dict:
    wr = random.Random(cfg.seed)
    tasks = [Task(i, (wr.uniform(0, AREA), wr.uniform(0, AREA)),
                  wr.uniform(0.5, 1.0) * cfg.deadline, cfg.service_time)
             for i in range(cfg.n_tasks)]
    kinds = ATTACKS[cfg.attack]
    mal = wr.sample(range(cfg.n_uavs), cfg.n_malicious) if kinds else []
    behaviors = {i: "honest" for i in range(cfg.n_uavs)}
    for k, uid in enumerate(mal):
        behaviors[uid] = kinds[k % len(kinds)]
    uavs = [UAV(i, behaviors[i], cfg) for i in range(cfg.n_uavs)]
    coord = Coordinator(uavs, tasks, cfg)
    if cfg.verbose:
        print("malicious UAVs:", {i: behaviors[i] for i in mal} or "none")

    now, end_t = 0.0, cfg.deadline
    while now <= cfg.deadline:
        msgs = []
        for u in uavs:
            msgs += u.step(now)
        coord.step(now, msgs)
        if coord.done:
            end_t = now
            break
        now += DT

    on_time = sum(1 for t in tasks if t.true_done and t.true_done_at <= t.deadline)
    true_done = sum(1 for t in tasks if t.true_done)
    claimed = sum(1 for t in tasks if t.status in ("claimed", "verified"))
    phantom = sum(1 for t in tasks if t.status in ("claimed", "verified") and not t.true_done)
    last_done = max([t.true_done_at for t in tasks if t.true_done], default=cfg.deadline)
    banned = set(coord.blacklist)
    mal_set = set(mal)
    return dict(
        success=on_time == cfg.n_tasks,
        on_time=on_time, true_done=true_done, claimed=claimed, phantom=phantom,
        mission_time=last_done if on_time == cfg.n_tasks else cfg.deadline,
        energy_kj=sum(u.energy_used for u in uavs) / 1000.0,
        crashed=sum(1 for u in uavs if not u.alive),
        tp=len(banned & mal_set), fn=len(mal_set - banned), fp=len(banned - mal_set),
        n_mal=len(mal_set),
        det_latency=[coord.blacklist[i] for i in banned & mal_set],
    )


def mean(xs):
    xs = list(xs)
    return sum(xs) / len(xs) if xs else float("nan")


def comparison(args):
    scenarios = [("no attack", "none", 0),
                 ("false battery", "false_battery", args.malicious),
                 ("spoofed msgs", "spoof_msg", args.malicious),
                 ("non-cooperative", "non_coop", args.malicious),
                 ("mixed (3 attackers)", "mixed", 3)]
    hdr = (f"{'scenario':<20}{'defense':<9}{'success%':>9}{'on-time':>9}{'phantom':>9}"
           f"{'energy kJ':>11}{'kJ/task':>9}{'time s':>8}{'crashed':>8}{'TP':>6}{'FP':>5}{'lat s':>7}")
    print(f"UAVs={args.uavs}  tasks={args.tasks}  deadline={args.deadline:.0f}s  seeds={args.seeds}")
    print("(phantom = tasks the coordinator believes done but were never done)\n")
    print(hdr)
    print("-" * len(hdr))
    for name, atk, nm in scenarios:
        for defense in (False, True):
            if atk == "none" and defense is False:
                pass
            rs = []
            for s in range(args.seeds):
                cfg = Config(n_uavs=args.uavs, n_tasks=args.tasks, deadline=args.deadline,
                             attack=atk, n_malicious=nm, defense=defense, seed=s)
                rs.append(run_sim(cfg))
            tp = sum(r["tp"] for r in rs)
            nmal = sum(r["n_mal" ] for r in rs)
            lat = [x for r in rs for x in r["det_latency"]]
            print(f"{name:<20}{('on' if defense else 'off'):<9}"
                  f"{100 * mean(r['success'] for r in rs):>9.0f}"
                  f"{mean(r['on_time'] for r in rs):>9.1f}"
                  f"{mean(r['phantom'] for r in rs):>9.1f}"
                  f"{mean(r['energy_kj'] for r in rs):>11.1f}"
                  f"{mean(r['energy_kj'] / max(1, r['true_done']) for r in rs):>9.2f}"
                  f"{mean(r['mission_time'] for r in rs):>8.0f}"
                  f"{mean(r['crashed'] for r in rs):>8.2f}"
                  f"{(f'{100 * tp / nmal:.0f}%' if nmal else '-'):>6}"
                  f"{sum(r['fp'] for r in rs):>5}"
                  f"{(f'{mean(lat):.0f}' if lat else '-'):>7}")
        print()


def main():
    p = argparse.ArgumentParser(description="UAV fleet coordination simulation")
    p.add_argument("--uavs", type=int, default=8)
    p.add_argument("--tasks", type=int, default=16)
    p.add_argument("--deadline", type=float, default=400.0)
    p.add_argument("--malicious", type=int, default=2)
    p.add_argument("--seeds", type=int, default=30)
    p.add_argument("--scenario", choices=list(ATTACKS), help="run ONE scenario with an event log")
    p.add_argument("--defense", action="store_true")
    p.add_argument("--seed", type=int, default=0)
    # Use parse_known_args() so that notebook/Colab kernel arguments (e.g., -f) are ignored.
    a, _ = p.parse_known_args()
    if a.scenario:
        cfg = Config(n_uavs=a.uavs, n_tasks=a.tasks, deadline=a.deadline, attack=a.scenario,
                     n_malicious=a.malicious, defense=a.defense, seed=a.seed, verbose=True)
        r = run_sim(cfg)
        print("\nresult:", {k: (round(v, 1) if isinstance(v, float) else v) for k, v in r.items()})
    else:
        comparison(a)


if __name__ == "__main__":
    main()


UAVs=8  tasks=16  deadline=400s  seeds=30
(phantom = tasks the coordinator believes done but were never done)

scenario            defense   success%  on-time  phantom  energy kJ  kJ/task  time s crashed    TP   FP  lat s
--------------------------------------------------------------------------------------------------------------
no attack           off            100     16.0      0.0       48.0     3.00      77    0.00     -    0      -
no attack           on             100     16.0      0.0       48.0     3.00      77    0.00     -    0      -

false battery       off              0     14.3      0.0       44.6     3.13     400    1.73    0%    0      -
false battery       on             100     16.0      0.0       51.3     3.21      93    0.47  100%    0     33

spoofed msgs        off              0     11.4      4.6       44.1     3.86     400    0.00    0%    0      -
spoofed msgs        on             100     16.0      0.0       53.4     3.34      96    0.00  100%    0     30